In [1]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi

PyTorch version: 2.10.0+cpu
CUDA available: False
/bin/bash: line 1: nvidia-smi: command not found


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content
!git clone https://github.com/chuangchuangtan/LGrad.git

/content
Cloning into 'LGrad'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 187 (delta 57), reused 123 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (187/187), 3.56 MiB | 48.54 MiB/s, done.
Resolving deltas: 100% (57/57), done.


In [4]:
%cd /content/LGrad

!pip install torch torchvision torchaudio --upgrade
!pip install opencv-python pillow tqdm scikit-learn
!pip install tensorboardX

/content/LGrad
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 8.6 MB/s eta 0:00:00


In [ ]:
!pip install -r requirements.txt

In [ ]:
%cd /content/LGrad/img2gad_pytorch

!wget https://lid-1302259812.cos.ap-nanjing.myqcloud.com/tmp/karras2019stylegan-bedrooms-256x256_discriminator.pth

/content/LGrad/img2gad_pytorch
--2026-02-24 11:45:54--  https://lid-1302259812.cos.ap-nanjing.myqcloud.com/tmp/karras2019stylegan-bedrooms-256x256_discriminator.pth
Resolving lid-1302259812.cos.ap-nanjing.myqcloud.com (lid-1302259812.cos.ap-nanjing.myqcloud.com)... 119.45.110.19, 119.45.110.23
Connecting to lid-1302259812.cos.ap-nanjing.myqcloud.com (lid-1302259812.cos.ap-nanjing.myqcloud.com)|119.45.110.19|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 92300155 (88M) [application/octet-stream]
Saving to: ‘karras2019stylegan-bedrooms-256x256_discriminator.pth’

karras2019stylegan- 100%[===================>]  88.02M  28.3MB/s    in 3.5s    

2026-02-24 11:45:58 (25.5 MB/s) - ‘karras2019stylegan-bedrooms-256x256_discriminator.pth’ saved [92300155/92300155]



In [ ]:
import fileinput

file_path = "/content/LGrad/img2gad_pytorch/gen_imggrad.py"

text = open(file_path).read()

text = text.replace(
"model.load_state_dict(torch.load(modelpath), strict=True)",
"model.load_state_dict(torch.load(modelpath, map_location='cpu'), strict=False)"
)

open(file_path,"w").write(text)

print("Fixed PyTorch compatibility")

Fixed PyTorch compatibility


In [ ]:
%cd /content/LGrad/img2gad_pytorch

!sh /content/LGrad/img2gad_pytorch/transform_img2grad.sh 0 \
/content/drive/MyDrive/LGrad/dataset \
/content/drive/MyDrive/LGrad/gradient

/content/LGrad/img2gad_pytorch
Transform /content/drive/MyDrive/LGrad/dataset/val/horse/0_real to /content/drive/MyDrive/LGrad/gradient/val/horse/0_real_grad
From /content/drive/MyDrive/LGrad/dataset/val/horse/0_real read 100 Img
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
[ WARN:0@5.557] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
Gen grad to /content/drive/MyDrive/LGrad/gradient/val/horse/0_real_grad/19929.png, bs:1 100/100
Transform /content/drive/MyDrive/LGrad/dataset/val/horse/1_fake to /content/drive/MyDrive/LGrad/gradient/val/horse/1_fake_grad
From /content/drive/MyDrive/LGrad/dataset/val/horse/1_fa

In [5]:
import os

os.makedirs("/content/drive/MyDrive/LGrad/checkpoints", exist_ok=True)

In [6]:
file = "/content/LGrad/CNNDetection/options/train_options.py"

text = open(file).read()

text = text.replace(
"self.parser.add_argument('--checkpoints_dir', type=str, default='./checkpoints'",
"self.parser.add_argument('--checkpoints_dir', type=str, default='/content/drive/MyDrive/LGrad/checkpoints'"
)

open(file,"w").write(text)

print("Checkpoint path modified to Drive")

Checkpoint path modified to Drive


In [5]:
!sed -i 's/python train.py/python train.py --num_threads 2/g' train-detector.sh

sed: can't read train-detector.sh: No such file or directory


In [4]:
%cd /content/LGrad/CNNDetection

!python train.py \
--name 4class-resnet-car-cat-chair-horse \
--dataroot /content/drive/MyDrive/LGrad/gradient \
--classes car,cat,chair,horse \
--batch_size 16 \
--lr 0.0005 \
--niter 50 \
--delr_freq 10 \
--checkpoints_dir /content/drive/MyDrive/LGrad/checkpoints

[Errno 2] No such file or directory: '/content/LGrad/CNNDetection'
/content
python3: can't open file '/content/train.py': [Errno 2] No such file or directory


In [1]:
!tree -L 3 /content/drive/MyDrive/LGrad/gradient/train

/bin/bash: line 1: tree: command not found


In [2]:
!for class in car cat chair horse; do \
rm -rf /content/drive/MyDrive/LGrad/gradient/train/$class/0_real_grad; \
rm -rf /content/drive/MyDrive/LGrad/gradient/train/$class/1_fake_grad; \
rm -rf /content/drive/MyDrive/LGrad/gradient/val/$class/0_real_grad; \
rm -rf /content/drive/MyDrive/LGrad/gradient/val/$class/1_fake_grad; \
done